In [27]:
import pandas as pd, json
from pathlib import Path

In [28]:
github_csv = pd.read_csv("projects.csv", dtype=str).fillna("")
github_csv["name_norm"] = github_csv["name"].str.lower().str.strip()

github_projects = [
    {
        "name": r["name"],
        "name_norm": r["name_norm"],
        "visibility": r.get("visibility", ""),
        "language": r.get("language", ""),
        "license": r.get("license", ""),
        "updated": r.get("updatedAt", "")
    }
    for _, r in github_csv.iterrows()
]

github_map = {p["name_norm"]: p for p in github_projects}


data = json.load(open("projects.json", "r", encoding="utf-8"))
prime = data.get("prime", {})

registry_projects = []
for category, items in prime.items():
    for it in items:
        ent = dict(it)
        ent["category"] = category
        ent["name_norm"] = ent.get("name", "").lower().strip()
        registry_projects.append(ent)

registry_map = {p["name_norm"]: p for p in registry_projects}

In [29]:
rows = []
registry_names = set(registry_map.keys())
github_names = set(github_map.keys())

for name_norm, reg in registry_map.items():
    gh = github_map.get(name_norm)  

    rows.append({
        "updated": gh["updated"] if gh else "",
        "category": reg.get("category"),
        "registry_name": reg.get("name"),
        "language": gh["language"] if gh else "",
        "status": reg.get("status", ""),

        "visibility": gh["visibility"] if gh else "",
        "github_name": gh["name"] if gh else "",
        "github_present": gh is not None,
    })

df = (
    pd.DataFrame(rows)
    .sort_values(["category", "registry_name"])
    .reset_index(drop=True)
)

matched = df[df["github_present"]]
only_registry = df[~df["github_present"]]
only_github = sorted(github_names - registry_names)

print(f"Registry JSON total: {len(registry_projects)}")
print(f"GitHub CSV total: {len(github_projects)}")
print(f"Matched (in both): {len(matched)}")
print(f"Only in registry JSON: {len(only_registry)}")
print(f"Only in GitHub CSV: {len(only_github)}")

Registry JSON total: 57
GitHub CSV total: 57
Matched (in both): 57
Only in registry JSON: 0
Only in GitHub CSV: 0


In [30]:
if only_github:
    print("Projects in GitHub CSV not found in registry JSON (normalized names):")
    display(pd.DataFrame({"name_norm": only_github}))


if not only_registry.empty:
    print("Projects in registry JSON but missing in GitHub CSV:")
    display(only_registry[["registry_name", "category"]])

In [31]:
display(df)

,updated,category,registry_name,language,status,visibility,github_name,github_present
0,2025-12-16 17:53:44,AI/ML,llmcolony,,experimental,PUBLIC,llmcolony,True
1,2025-04-23 20:22:55,AI/ML,simtube,,failed,PUBLIC,simtube,True
2,2025-12-10 11:41:43,AI/ML,telegramllmbot,,successful,PUBLIC,telegramllmbot,True
3,2026-02-10 09:46:48,Data,sparklineo,,-,PRIVATE,sparklineo,True
4,2025-03-23 23:11:24,Games,echecs,,failed,PUBLIC,echecs,True
5,2026-02-12 09:32:04,Games,games,,active,PUBLIC,games,True
6,2026-02-12 11:07:04,Games,games.py,,active,PUBLIC,games.py,True
7,2026-02-24 10:39:56,Games,games.rs,,-,PUBLIC,games.rs,True
8,2025-03-13 07:38:10,Games,hunch,,merged,PUBLIC,hunch,True
9,2025-03-24 02:13:19,Games,tictactoe,,-,PUBLIC,tictactoe,True
